# Axion Signal Detection Simulation

This notebook simulates weak axion-like signals embedded in Gaussian noise and studies detection strategies using frequency-domain methods, threshold calibration, and semi-coherent analysis.

## Tools
- Python
- NumPy
- Matplotlib
- SciPy

## Goal
To model weak signal detection in noisy environments and evaluate how detection sensitivity changes with integration time and coherence assumptions.

In [1]:
import numpy as np

# 1) settings
fs = 10_000
T = 10
N = fs * T
t = np.arange(N) / fs

# 2) noise
sigma = 1.0
noise = np.random.normal(0, sigma, size=N)

# 3) injected weak signal
A = 0.2
f0 = 1234
sig = A * np.sin(2*np.pi*f0*t)

x_noise = noise
x_sig = noise + sig

# 4) PSD via FFT
def psd(x):
    X = np.fft.rfft(x)
    P = (np.abs(X)**2) / N
    f = np.fft.rfftfreq(N, d=1/fs)
    return f, P

f, Pn = psd(x_noise)
_, Ps = psd(x_sig)

# 5) simple detection test
# ignore DC bin for threshold
mu = np.mean(Pn[1:])
std = np.std(Pn[1:])
k = 6  # stricter threshold
thr = mu + k*std

peak_idx = np.argmax(Ps[1:]) + 1
detected = Ps[peak_idx] > thr

print("Peak frequency:", f[peak_idx])
print("Peak power:", Ps[peak_idx])
print("Threshold:", thr)
print("Detected?", detected)


Peak frequency: 1234.0
Peak power: 1089.2044067142988
Threshold: 6.987393047221699
Detected? True


In [2]:
import numpy as np

fs = 10_000
T = 60
N = fs * T
t = np.arange(N) / fs

sigma = 1.0
noise = np.random.normal(0, sigma, size=N)

A = 0.05
f0 = 1234.0
Tc = 0.5  # coherence time in seconds
chunk_N = int(Tc * fs)
num_chunks = N // chunk_N

# Build a "semi-coherent" signal: phase resets each chunk
sig = np.zeros(N)
for i in range(num_chunks):
    start = i * chunk_N
    end = start + chunk_N
    tt = t[start:end]
    phi = np.random.uniform(0, 2*np.pi)

    # Optional small frequency jitter (uncomment to add)
    # df = np.random.normal(0, 0.5)  # Hz jitter
    # f = f0 + df
    f = f0

    sig[start:end] = A * np.sin(2*np.pi*f*tt + phi)

x = noise + sig

# PSD (single FFT)
def psd_fft(x):
    X = np.fft.rfft(x)
    P = (np.abs(X)**2) / N
    f = np.fft.rfftfreq(N, d=1/fs)
    return f, P

# Welch PSD: average PSD across segments
def psd_welch(x, seg_N):
    step = seg_N  # non-overlapping for simplicity
    M = (len(x) - seg_N) // step + 1
    P_accum = None
    for m in range(M):
        s = m * step
        e = s + seg_N
        xx = x[s:e]
        X = np.fft.rfft(xx)
        P = (np.abs(X)**2) / seg_N
        if P_accum is None:
            P_accum = P
        else:
            P_accum += P
    P_avg = P_accum / M
    f = np.fft.rfftfreq(seg_N, d=1/fs)
    return f, P_avg

# Detection helper
def detect_peak(f, P, k=6):
    mu = np.mean(P[1:])
    std = np.std(P[1:])
    thr = mu + k*std
    peak_idx = np.argmax(P[1:]) + 1
    return f[peak_idx], P[peak_idx], thr, (P[peak_idx] > thr)

# Compute and detect
f1, P1 = psd_fft(x)
pkf1, pkp1, thr1, det1 = detect_peak(f1, P1, k=6)

# Welch using segment length = Tc (matches coherence chunks)
fw, Pw = psd_welch(x, seg_N=chunk_N)
pkfw, pkpw, thrw, detw = detect_peak(fw, Pw, k=6)

print("FFT:   peak f=", pkf1, "peak P=", pkp1, "thr=", thr1, "detected=", det1)
print("Welch: peak f=", pkfw, "peak P=", pkpw, "thr=", thrw, "detected=", detw)


FFT:   peak f= 1233.3666666666666 peak P= 24.4423615663519 thr= 7.044082412475197 detected= True
Welch: peak f= 1234.0 peak P= 4.322949985077883 thr= 1.6733066473600913 detected= True


In [3]:
import numpy as np

# ---------- settings ----------
fs = 10_000
T = 60
N = fs * T
t = np.arange(N) / fs

sigma = 1.0
A = 0.05
f0 = 1234.0
Tc = 0.5
chunk_N = int(Tc * fs)

# ---------- build semi-coherent signal ----------
def make_semicoherent_signal(t, f0, A, fs, Tc):
    N = len(t)
    chunk_N = int(Tc * fs)
    num_chunks = N // chunk_N
    sig = np.zeros(N)
    for i in range(num_chunks):
        s = i * chunk_N
        e = s + chunk_N
        tt = t[s:e]
        phi = np.random.uniform(0, 2*np.pi)
        sig[s:e] = A * np.sin(2*np.pi*f0*tt + phi)
    return sig

# ---------- Welch PSD ----------
def psd_welch(x, fs, seg_N):
    step = seg_N  # non-overlapping
    M = (len(x) - seg_N) // step + 1
    P_accum = None
    for m in range(M):
        s = m * step
        e = s + seg_N
        xx = x[s:e]
        X = np.fft.rfft(xx)
        P = (np.abs(X)**2) / seg_N
        if P_accum is None:
            P_accum = P
        else:
            P_accum += P
    P_avg = P_accum / M
    f = np.fft.rfftfreq(seg_N, d=1/fs)
    return f, P_avg

# ---------- helper: peak in a frequency window ----------
def peak_in_window(f, P, f_center, half_width_hz):
    mask = (f >= f_center - half_width_hz) & (f <= f_center + half_width_hz)
    idxs = np.where(mask)[0]
    j = idxs[np.argmax(P[idxs])]
    return f[j], P[j]

# ---------- 1) simulate signal+noise and measure peak near f0 ----------
noise = np.random.normal(0, sigma, size=N)
sig = make_semicoherent_signal(t, f0, A, fs, Tc)
x = noise + sig

f, P = psd_welch(x, fs, seg_N=chunk_N)
fpk_obs, Ppk_obs = peak_in_window(f, P, f0, half_width_hz=2.0)

print("Observed (signal+noise): peak f =", fpk_obs, "peak P =", Ppk_obs)

# ---------- 2) Monte Carlo noise-only to estimate false alarm probability ----------
trials = 200
count = 0
for _ in range(trials):
    n = np.random.normal(0, sigma, size=N)
    ff, PP = psd_welch(n, fs, seg_N=chunk_N)
    _, Ppk = peak_in_window(ff, PP, f0, half_width_hz=2.0)
    if Ppk >= Ppk_obs:
        count += 1

FAP = count / trials
print("False alarm probability (in ±2 Hz window) =", FAP)

# ---------- decision ----------
alpha = 0.01  # 1%
decision = "DETECTION" if FAP < alpha else "NOT significant"
print("Decision:", decision, "(alpha =", alpha, ")")


Observed (signal+noise): peak f = 1234.0 peak P = 4.487802359176234
False alarm probability (in ±2 Hz window) = 0.0
Decision: DETECTION (alpha = 0.01 )


In [4]:
import numpy as np

fs = 10_000
T = 60
N = fs * T
t = np.arange(N) / fs

sigma = 1.0
A = 0.05
f0 = 1234.0
Tc = 0.5
seg_N = int(Tc * fs)

# --- signal generator (phase resets each segment) ---
def make_semicoherent_signal(t, f0, A, fs, Tc):
    N = len(t)
    seg_N = int(Tc * fs)
    M = N // seg_N
    sig = np.zeros(N)
    for m in range(M):
        s = m * seg_N
        e = s + seg_N
        tt = t[s:e]
        phi = np.random.uniform(0, 2*np.pi)
        sig[s:e] = A * np.sin(2*np.pi*f0*tt + phi)
    return sig

# --- Welch PSD ---
def psd_welch(x, fs, seg_N):
    step = seg_N
    M = (len(x) - seg_N) // step + 1
    P_accum = None
    for m in range(M):
        s = m * step
        e = s + seg_N
        xx = x[s:e]
        X = np.fft.rfft(xx)
        P = (np.abs(X)**2) / seg_N
        if P_accum is None:
            P_accum = P
        else:
            P_accum += P
    P_avg = P_accum / M
    f = np.fft.rfftfreq(seg_N, d=1/fs)
    return f, P_avg

# --- peak statistic in a window: z = (peak - mean)/std ---
def window_stat(f, P, f_center, half_width_hz):
    mask = (f >= f_center-half_width_hz) & (f <= f_center+half_width_hz)
    idx = np.where(mask)[0]
    Pw = P[idx]
    peak = np.max(Pw)
    mu = np.mean(Pw)
    sd = np.std(Pw) + 1e-12
    z = (peak - mu) / sd
    fpk = f[idx[np.argmax(Pw)]]
    return fpk, peak, z

# --- simulate data ---
noise = np.random.normal(0, sigma, size=N)
sig = make_semicoherent_signal(t, f0, A, fs, Tc)
x = noise + sig

f, P = psd_welch(x, fs, seg_N=seg_N)

# --- scan settings ---
f_start, f_stop, df = 1000.0, 1500.0, 1.0
centers = np.arange(f_start, f_stop + df, df)
N_tests = len(centers)

alpha_global = 0.01
alpha_local = alpha_global / N_tests  # Bonferroni

# Convert alpha_local to a z-threshold approximately:
# For simplicity we use a fixed high z threshold; you’ll calibrate it next step.
# Start with z_thr = 6 and then we’ll calibrate using noise-only simulations.
z_thr = 6.0

candidates = []
for fc in centers:
    fpk, peak, z = window_stat(f, P, fc, half_width_hz=2.0)
    if z > z_thr:
        candidates.append((fc, fpk, peak, z))

print("Scan tests:", N_tests, "alpha_local:", alpha_local, "z_thr:", z_thr)
print("Candidates found:", len(candidates))
print("First 10 candidates (fc, fpk, peak, z):")
for row in candidates[:10]:
    print(row)

# sanity: show nearest candidate to true f0 if exists
if candidates:
    nearest = min(candidates, key=lambda r: abs(r[0] - f0))
    print("Nearest candidate to f0:", nearest)


Scan tests: 501 alpha_local: 1.9960079840319362e-05 z_thr: 6.0
Candidates found: 0
First 10 candidates (fc, fpk, peak, z):


In [5]:
import numpy as np

fs = 10_000
T = 60
N = fs * T
t = np.arange(N) / fs

sigma = 1.0
A = 0.05
f0 = 1234.0
Tc = 0.5
seg_N = int(Tc * fs)

# --- signal generator (phase resets each segment) ---
def make_semicoherent_signal(t, f0, A, fs, Tc):
    N = len(t)
    seg_N = int(Tc * fs)
    M = N // seg_N
    sig = np.zeros(N)
    for m in range(M):
        s = m * seg_N
        e = s + seg_N
        tt = t[s:e]
        phi = np.random.uniform(0, 2*np.pi)
        sig[s:e] = A * np.sin(2*np.pi*f0*tt + phi)
    return sig

# --- Welch PSD ---
def psd_welch(x, fs, seg_N):
    step = seg_N
    M = (len(x) - seg_N) // step + 1
    P_accum = None
    for m in range(M):
        s = m * step
        e = s + seg_N
        xx = x[s:e]
        X = np.fft.rfft(xx)
        P = (np.abs(X)**2) / seg_N
        if P_accum is None:
            P_accum = P
        else:
            P_accum += P
    P_avg = P_accum / M
    f = np.fft.rfftfreq(seg_N, d=1/fs)
    return f, P_avg

# --- robust line statistic in a window ---
def R_stat_in_window(f, P, f_center, half_width_hz):
    mask = (f >= f_center-half_width_hz) & (f <= f_center+half_width_hz)
    idx = np.where(mask)[0]
    Pw = P[idx]
    j = idx[np.argmax(Pw)]
    peak = P[j]
    med = np.median(Pw) + 1e-18
    R = peak / med
    return f[j], peak, R

# --- scan over a frequency range, return max R and where it occurred ---
def scan_max_R(f, P, f_start=1000.0, f_stop=1500.0, df=1.0, half_width_hz=2.0):
    centers = np.arange(f_start, f_stop+df, df)
    best = None  # (R, fc, fpk, peak)
    for fc in centers:
        fpk, peak, R = R_stat_in_window(f, P, fc, half_width_hz)
        if (best is None) or (R > best[0]):
            best = (R, fc, fpk, peak)
    return best  # (Rmax, fc_center, f_peak, peak_power)

# ---------- calibration: noise-only ----------
trials = 200
Rmax_list = []

for _ in range(trials):
    n = np.random.normal(0, sigma, size=N)
    ff, PP = psd_welch(n, fs, seg_N=seg_N)
    Rmax, fc, fpk, peak = scan_max_R(ff, PP)
    Rmax_list.append(Rmax)

Rmax_list = np.array(Rmax_list)
alpha_global = 0.01
thr = np.quantile(Rmax_list, 1 - alpha_global)

print("Calibration done.")
print("Noise-only max R: mean =", Rmax_list.mean(), "std =", Rmax_list.std())
print("Global threshold (alpha=1%) R_thr =", thr)

# ---------- test: signal+noise ----------
noise = np.random.normal(0, sigma, size=N)
sig = make_semicoherent_signal(t, f0, A, fs, Tc)
x = noise + sig

f, P = psd_welch(x, fs, seg_N=seg_N)
Rmax, fc_best, fpk_best, peak_best = scan_max_R(f, P)

detected = Rmax > thr
print("\nSignal+noise scan result:")
print("R_max =", Rmax, "at center fc =", fc_best, "peak freq =", fpk_best)
print("Detected?", detected)


Calibration done.
Noise-only max R: mean = 1.3642771222774976 std = 0.06231555108278391
Global threshold (alpha=1%) R_thr = 1.5513249598950614

Signal+noise scan result:
R_max = 3.8123002638560917 at center fc = 1232.0 peak freq = 1234.0
Detected? True


In [2]:
import numpy as np

fs = 10_000
sigma = 1.0
f0 = 1234.0
Tc = 0.5
alpha_global = 0.01

Ts = [10, 20, 40, 80, 160]   # integration times (s)
A_grid = np.linspace(0.01, 0.12, 12)  # trial amplitudes

# --- functions reused ---
def make_semicoherent_signal(t, f0, A, fs, Tc):
    N = len(t)
    seg_N = int(Tc * fs)
    M = N // seg_N
    sig = np.zeros(N)
    for m in range(M):
        s = m * seg_N
        e = s + seg_N
        tt = t[s:e]
        phi = np.random.uniform(0, 2*np.pi)
        sig[s:e] = A * np.sin(2*np.pi*f0*tt + phi)
    return sig

def psd_welch(x, fs, seg_N):
    step = seg_N
    M = (len(x) - seg_N) // step + 1
    P_accum = None
    for m in range(M):
        s = m * step
        e = s + seg_N
        xx = x[s:e]
        X = np.fft.rfft(xx)
        P = (np.abs(X)**2) / seg_N
        if P_accum is None:
            P_accum = P
        else:
            P_accum += P
    P_avg = P_accum / M
    f = np.fft.rfftfreq(seg_N, d=1/fs)
    return f, P_avg

def R_stat_in_window(f, P, f_center, half_width_hz=2.0):
    mask = (f >= f_center-half_width_hz) & (f <= f_center+half_width_hz)
    idx = np.where(mask)[0]
    Pw = P[idx]
    j = idx[np.argmax(Pw)]
    peak = P[j]
    med = np.median(Pw) + 1e-18
    R = peak / med
    return R

def scan_max_R(f, P, f_start=1000, f_stop=1500, df=1.0):
    centers = np.arange(f_start, f_stop+df, df)
    Rmax = 0
    for fc in centers:
        R = R_stat_in_window(f, P, fc)
        if R > Rmax:
            Rmax = R
    return Rmax

# --- main loop ---
results = []

for T in Ts:
    N = int(fs * T)
    t = np.arange(N) / fs
    seg_N = int(Tc * fs)

    # calibrate threshold
    trials = 100
    Rmax_noise = []
    for _ in range(trials):
        n = np.random.normal(0, sigma, size=N)
        f, P = psd_welch(n, fs, seg_N)
        Rmax_noise.append(scan_max_R(f, P))
    R_thr = np.quantile(Rmax_noise, 1 - alpha_global)

    # find minimum A that exceeds threshold
    A_detect = None
    for A in A_grid:
        n = np.random.normal(0, sigma, size=N)
        sig = make_semicoherent_signal(t, f0, A, fs, Tc)
        x = n + sig
        f, P = psd_welch(x, fs, seg_N)
        Rmax = scan_max_R(f, P)
        if Rmax > R_thr:
            A_detect = A
            break

    results.append((T, R_thr, A_detect))
    print(f"T={T}s | R_thr={R_thr:.2f} | A_min={A_detect}")

print("\nResults:", results)


T=10s | R_thr=2.89 | A_min=0.03
T=20s | R_thr=2.08 | A_min=0.04
T=40s | R_thr=1.68 | A_min=0.03
T=80s | R_thr=1.43 | A_min=0.02
T=160s | R_thr=1.28 | A_min=0.02

Results: [(10, 2.8896540483891027, 0.03), (20, 2.0805094541872373, 0.04), (40, 1.6842531837708015, 0.03), (80, 1.4313991413353093, 0.02), (160, 1.2779631254880646, 0.02)]


In [7]:
import numpy as np

fs = 10_000
sigma = 1.0
f0 = 1234.0
alpha_global = 0.01

Ts = [10, 20, 40, 80, 160]
Tc_list = [0.1, 0.5, 1.0, 2.0]
A_grid = np.linspace(0.01, 0.12, 12)

def make_semicoherent_signal(t, f0, A, fs, Tc):
    N = len(t)
    seg_N = int(Tc * fs)
    if seg_N < 16:
        seg_N = 16
    M = N // seg_N
    sig = np.zeros(N)
    for m in range(M):
        s = m * seg_N
        e = s + seg_N
        tt = t[s:e]
        phi = np.random.uniform(0, 2*np.pi)
        sig[s:e] = A * np.sin(2*np.pi*f0*tt + phi)
    return sig

def psd_welch(x, fs, seg_N):
    step = seg_N
    M = (len(x) - seg_N) // step + 1
    P_accum = None
    for m in range(M):
        s = m * step
        e = s + seg_N
        xx = x[s:e]
        X = np.fft.rfft(xx)
        P = (np.abs(X)**2) / seg_N
        if P_accum is None:
            P_accum = P
        else:
            P_accum += P
    P_avg = P_accum / M
    f = np.fft.rfftfreq(seg_N, d=1/fs)
    return f, P_avg

def R_stat_in_window(f, P, f_center, half_width_hz=2.0):
    mask = (f >= f_center-half_width_hz) & (f <= f_center+half_width_hz)
    idx = np.where(mask)[0]
    
    # Check if idx is empty and handle this case
    if len(idx) == 0:
        return 1.0  # Return neutral R value if no frequencies in the window
        
    Pw = P[idx]
    j = idx[np.argmax(Pw)]
    peak = P[j]
    med = np.median(Pw) + 1e-18
    return peak / med

def scan_max_R(f, P, f_start=1000, f_stop=1500, df=1.0):
    centers = np.arange(f_start, f_stop+df, df)
    Rmax = 0.0
    for fc in centers:
        R = R_stat_in_window(f, P, fc)
        if R > Rmax:
            Rmax = R
    return Rmax

all_results = {}

for Tc in Tc_list:
    print("\n==== Tc =", Tc, "s ====")
    Tc_results = []
    for T in Ts:
        N = int(fs * T)
        t = np.arange(N) / fs
        seg_N = int(Tc * fs)
        if seg_N < 16:
            seg_N = 16

        # Calibrate global threshold
        trials = 100
        Rmax_noise = []
        for _ in range(trials):
            n = np.random.normal(0, sigma, size=N)
            f, P = psd_welch(n, fs, seg_N)
            Rmax_noise.append(scan_max_R(f, P))
        R_thr = np.quantile(Rmax_noise, 1 - alpha_global)

        # Find A_min
        A_min = None
        for A in A_grid:
            n = np.random.normal(0, sigma, size=N)
            sig = make_semicoherent_signal(t, f0, A, fs, Tc)
            x = n + sig
            f, P = psd_welch(x, fs, seg_N)
            Rmax = scan_max_R(f, P)
            if Rmax > R_thr:
                A_min = A
                break

        Tc_results.append((T, R_thr, A_min))
        print(f"T={T:>3}s | R_thr={R_thr:.2f} | A_min={A_min}")

    all_results[Tc] = Tc_results

print("\nDone. all_results dict contains everything.")



==== Tc = 0.1 s ====
T= 10s | R_thr=1.00 | A_min=None
T= 20s | R_thr=1.00 | A_min=None
T= 40s | R_thr=1.00 | A_min=None
T= 80s | R_thr=1.00 | A_min=None
T=160s | R_thr=1.00 | A_min=None

==== Tc = 0.5 s ====
T= 10s | R_thr=2.64 | A_min=0.04
T= 20s | R_thr=1.99 | A_min=0.03
T= 40s | R_thr=1.64 | A_min=0.03
T= 80s | R_thr=1.44 | A_min=0.03
T=160s | R_thr=1.31 | A_min=0.02

==== Tc = 1.0 s ====
T= 10s | R_thr=4.23 | A_min=0.05
T= 20s | R_thr=2.65 | A_min=0.03
T= 40s | R_thr=1.92 | A_min=0.02
T= 80s | R_thr=1.62 | A_min=0.02
T=160s | R_thr=1.45 | A_min=0.01

==== Tc = 2.0 s ====
T= 10s | R_thr=5.40 | A_min=0.03
T= 20s | R_thr=3.50 | A_min=0.03
T= 40s | R_thr=2.53 | A_min=0.02
T= 80s | R_thr=1.96 | A_min=0.02
T=160s | R_thr=1.62 | A_min=0.02

Done. all_results dict contains everything.


## Conclusion

The simulation shows that weak axion-like signals can become detectable under longer integration times and appropriate threshold calibration. The notebook also illustrates how coherence assumptions affect sensitivity and candidate recovery in noisy data.